This example demonstrates how to run the STA30 example assuming that you have sucessfully downloaded the code and installed the necessary libraries. The first step is to obtain an extended xyz file of the structure. The mosr conveneint way is to access the ICSD and download the cif file (it is not included here because of copyright but the file, cif77_opt.xyz is included and the next step can be skipped if you want and just copy the file to basin.xyz)

In [5]:
from ase import Atoms
from ase.io import read, write
import numpy as np

def _substitute_al(symbols):
    """
    substitute Al for Si atom
    """
    slist = []
    for i in range(len(symbols)):
        if symbols[i] == 'Si':
            slist.append(i)

    for i in range(10):
        num = int(len(slist) * np.random.random())

        choice = slist[num]
        slist.pop(num)

        symbols[choice] = "Al"

atoms_object = read("EntryWithCollCode122377.cif")
_substitute_al(atoms_object.symbols)
write(filename="basin.xyz", images=atoms_object, format="extxyz")

Once the basin.xyz has been created then the appropriate number, 10 in this case, Si atoms must be substituted by Al to make the cell charge neutral. This can easily be done by hand using a text editor such as vi or emacs. Once the basin.xyz has been built then it is possible to start running the code to generate configurations. In this example we are using the MACE_mp NKIP's and it is necessary to download a suitable library. An off-the-shelf solution is MACE-MATPES-r2SCAN-0 obtained from ACEsuit (https://github.com/ACEsuit/mace).

To execute the program two additional files are needed; the control file to provide commands to the program and potentials to provide parameters.

A basic control file for STA30 can be created by running the python script

In [1]:

# Sample code to generate output
sample_list = ['method basinhop', 'steps 5', 'eqilsteps 2', 'temperature 1300', 'move swap 1 100', 'Al Si', 'close']

for item in sample_list:
    print(item)
# Save the captured output to a text file
file = open('control', 'w')
for item in sample_list:
    file.write(item)
    file.write("\n")

file.flush()
file.close()


method basinhop
steps 5
eqilsteps 2
temperature 1300
move swap 1 100
Al Si
close


The next step is to create the potentials file assuming that the mace model is in the working directory.

In [3]:
sample_list = ['species 4', 'K 0.0 0.0 13', 'Al 0.0 0.0 13', 'Si 0.0 0.0 14', 'O 0.0 0.0 8', 'model MACE-matpes-r2scan-omat-ft.model', 'close']

for item in sample_list:
    print(item)
# Save the captured output to a text file
file = open('potentials', 'w')
for item in sample_list:
    file.write(item)
    file.write("\n")

file.flush()
file.close()

species 4
K 0.0 0.0 13
Al 0.0 0.0 13
Si 0.0 0.0 14
O 0.0 0.0 8
model MACE-matpes-r2scan-omat-ft.model
close


To run the calculation it is probably best to quit the jupyter notebook and transfer the files to a cluster style computer where GPU's are available, For example, on the SCARF computer at RAL where the calculations can be run either interactively or by the submission of a suitable script. Although most clusters will be subtly different, a slurm script is given below as a template

#!/bin/bash
#SBATCH --job-name=kmcCoB
#SBATCH -t 24:00:00
#SBATCH -C amd
#SBATCH -p gpu
#SBATCH --gres=gpu:1
#SBATCH --nodes=1
#SBATCH -o %Jtrainout.log
#SBATCH -e %J.err
#SBATCH --ntasks-per-node=1
#SBATCH --exclusive

ml Python
ml mpi4py

export PYTHONPATH="${PYTHONPATH}:/home/vol05/scarf145/.local/bin"

mpirun -np 1 python3 /home/vol05/scarf145/pymc-nmr/src/monte.py



 After the simulation you should open the file mc.log. Towards the botton of the file you should find something along the lines of:

 *****************************************************************************************************
 final energy of system containing 226 atoms
 *****************************************************************************************************


 energies of box 1
 total (internal) energy         -2.205519473822256e+03

 final sanity check

 energies of box 0
 total (internal) energy          0.000000000000000e+00

 Dont worry if the total energy is different from the results above as it is stochastic, however, you should make sure that the samity check gives a zero energy.

 Further details of the simulation are given in the section:

 *****************************************************************************************************
 Summary of simulation
 *****************************************************************************************************
 

 swaps Al Si: attempted, successful and ratio 5.0 1.0 2.0000000000e-01

 swaps Al Si: Downhill and Uphill 1.0 0.0

 This part indicates how efficient the method employed, in this instance basin hopping, is at swapping Si/Al around. The simulation was only for 5 steps so you may not get any swaps at all. Just run it for longer. Downhill moves are accepted swaps where the energy is less than the energy of the previous configuration. Uphill swaps is where the energy is greater than the previous configuration yet has been excepted via Boltzmann sampling.

 It is also possible to examine the structures produced by the simulation thos configurations accepted via the sampling can be found in accepted.xyz whilst the restart.xyz provides the final configuration (it is possible to use something like jmol to display the structure).




The above example demonstrates the basic approach to basin hopping. However, it is possible to alter the parameters for the energy minimisation through the control file and you should consult the user guide. For example, the method of relaxation can be changed with the key-word relmethod

In some instances where just swapping atoms is inhibited, at least partialy by the presence of charge balancing cations and or vacancies. Unfortunately, it is not always possible know where a suitable position is located. Thus it is possible to do a short NVT molecular dynamics. The easiest way to do this is to open the control file in a text editor (vi or emacs) and add the following line:

move moldyn 20

The integer gives the frequency or weighting that a molecular dynamics simulation will be chosen. In the previus example the frequency for the swap move was 100. MB the total frequency does not need to add up to 100 as the weighting is relative - 20 / 120 in this case. The temperature, number of steps of the MD can be adjusted indepedently of the basin hopping - see the manual for keyword.

In the summary section you will see an additional line:

the number of MD runs 1 (NB this wont appear if ther have been no MD calculations selected in this very short run)

In the final part of this example, the use of the combined swap or combiswap is discussed. The combined swap moves 3 atoms A B and C. This is a five step process:
1. A grid is constructed and the grid point is designated as occupied if it is within a certain distance of an atom or unoccupied if it is greater than the specified distance.
2. Atoms of type A and B are selected at random
3. The atom of type C that is closest to the randomly selected B atom is identified.
4. Atoms and and B are swapped. The location of the C atom is obtained from the grid by determining the unoccupied grid points within a distance of the B atom. 
5. The C atom is then given the position of one of these empty grid points and the system is relaxed.

The size of the grid is determined by the grid key-word (e.g. grid 10 10 10 2.0). The three integers are the number of required intervals acoss the x,y,a nd z dimensions of the cell. The float is the distance used to determine whether a grid point is occupied or unoccupied. 
The combined swap is activated in the instance of the STA30 example by 

move combiswap 1 100 3.5
Si Al K 

The first integer is the number of combiswaps required and the second the wighting for the occurance. The float is used to evaluate empty grid points in step 4. The combined swap in this instance is Si and Al with the K following the Al atom (the order here is important).

In [1]:
# Sample code to generate control file for combiswap
sample_list = ['method basinhop', 'steps 5', 'eqilsteps 2', 'temperature 1300', 'grid 10 10 10 2.0', 'move combiswap 1 100 3.5', 'Si Al K', 'close']

for item in sample_list:
    print(item)
# Save the captured output to a text file
file = open('control', 'w')
for item in sample_list:
    file.write(item)
    file.write("\n")

file.flush()
file.close()

method basinhop
steps 5
eqilsteps 2
temperature 1300
grid 10 10 10 2.0
move combiswap 1 100 3.5
Si Al K
close


The type of swap required and whether intermittent MD runs are required depends on the type of atoms being swapped and the chemical makeup of the simulation cell.